In [1]:
import os
from datetime import date
from dotenv import load_dotenv
from functions import login, buy_etf, get_free_cash, load_allocations, send_email

### Load variables

In [2]:
# Load environment file. The secrets live in `_env` in the project root
# (one level above src/); fall back to a few common names/locations so the
# notebook works whether it is run from src/ or the project root.
for candidate in ["_env", "../_env", ".env", "../.env"]:
    if os.path.exists(candidate):
        load_dotenv(candidate, override=True)
        break

today = date.today().strftime("%Y-%m-%d")

# Access variables
USERNAME = os.getenv("EE_USERNAME")
PASSWORD = os.getenv("EE_PASSWORD")
account_id = os.getenv("ZAR_TRADING_ACCOUNT_ID")
MIN_TRADE = float(os.getenv("MIN_TRADE") or 10)
EMAIL_SENDER = os.getenv("EMAIL_SENDER")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
EMAIL_RECEIVER = os.getenv("EMAIL_RECEIVER")

assert USERNAME and PASSWORD, "EE_USERNAME / EE_PASSWORD not found — check your _env file."
assert account_id, "ZAR_TRADING_ACCOUNT_ID not found — check your _env file."

### Connect/Login to Easy Equities

In [4]:
# EasyEquities moved to an OAuth2/OpenID Connect login (identity.openeasy.io),
# so the library's built-in client.login() no longer works. Our login() helper
# drives the new flow and returns an authenticated client.
client = login(USERNAME, PASSWORD)
print("Logged in successfully.")

Logged in successfully.


### Determine free cash available to invest

In [5]:
# get free cash allocations on selected account
free_cash = get_free_cash(client, account_id)

### Determine allocation ammounts

In [6]:
# get the ticker, amount allocations
allocation_amounts = load_allocations(free_cash)

### Place buy orders

In [7]:
# iterate through all tickers  and execute buy trades
results = []
for ticker, data in allocation_amounts.items():
    amount = data["amount"]
    contract_code = data["contract_code"]
    ticker_name = data["ticker_name"]
    try:
        if amount < MIN_TRADE:
            results.append(f"SKIP {ticker_name}: Amount too small (R{amount:.2f})")
            continue

        # place buy order
        response = buy_etf(client, account_id, contract_code, amount)
        if response.status_code == 200:
            # Add results
            results.append(f"BUY {ticker_name}: R{amount:.2f} ✅")
        else:
            results.append(f"Failed to place order {ticker_name}: {response.text}")
    except Exception as e:
        results.append(f"Failed to place buy order for {ticker_name}:{e}")

### Send email with result of trade

In [9]:
# Send an email summary of the trades, if email is configured. Otherwise just
# print the results so the notebook still works without email set up.
print("\n".join(results) if results else "No results.")

if EMAIL_SENDER and EMAIL_PASSWORD and EMAIL_RECEIVER:
    send_email(results, today, EMAIL_SENDER, EMAIL_RECEIVER, EMAIL_PASSWORD)
    print("\nEmail sent.")
else:
    print("\nEmail not configured (EMAIL_* vars blank) — skipping email notification.")

BUY "Satrix S&P 500 ETF": R12.10 ✅
BUY "EasyCrypto 10": R12.10 ✅
BUY "New Wave Silver ETN": R12.10 ✅
BUY "NewPlat ETF": R12.10 ✅
BUY "Satrix 40 ETF": R12.10 ✅

Email sent.
